# CSE475 - Phase 1 baseline (Colab T4, GPU)

Same portable harness as Kaggle:
- dataset from HF (`Neperl/skin-disease-acne-rosacea-normal`)
- checkpoint synced to HF (`Nirob-jon/cse475-skin-checkpoints`) after every epoch
- resumes from the latest HF checkpoint automatically

**Setup required:**
1. Runtime -> Change runtime type -> **GPU (T4)**
2. Left panel -> Secrets -> add a write-scope HF token as `HF_TOKEN`
3. Run all. The last cell mounts Drive (used for checkpoints via the `colab_t4` profile).

In [ ]:
import os, getpass
from huggingface_hub import login, snapshot_download

HF_REPO = "Nirob-jon/cse475-skin-checkpoints"

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Paste your HF token (hf_...): ")
assert token, "No HF_TOKEN found"
login(token=token, add_to_git_credential=False)
print("logged in OK")

In [ ]:
# clone the harness (public repo)
!git clone --depth 1 https://github.com/nirjon001/multi-source-skin-disease-fusion.git
%cd multi-source-skin-disease-fusion

In [ ]:
# dataset
DATA = snapshot_download("Neperl/skin-disease-acne-rosacea-normal", repo_type="dataset")
print("dataset at:", DATA)
import os
print(sorted(os.listdir(os.path.join(DATA, "train"))))

In [ ]:
# light deps only (torch/torchvision preinstalled on Colab)
!pip install -q timm tqdm pyyaml imagehash scikit-learn huggingface_hub

In [ ]:
!python src/audit.py --data "$DATA" --check-corrupt --out results/audit_starter.json

## Mount Drive (for checkpoints)

The `colab_t4` profile saves checkpoints to `/content/drive/MyDrive/cse475/results`.
HF sync also runs after every epoch, so Drive is a second copy - it also survives
the occasional Colab session reset.

If you would rather NOT use Drive: change `--profile colab_t4` to `--profile kaggle_t4`
in the training cell (that profile writes to `/kaggle/working/...` which still works on
Colab because it's just a folder path, and HF sync covers the rest).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# TRAIN (Colab profile: CUDA torch, batch 32, AMP on, checkpoints to Drive)
!python src/train_resumable.py --config configs/baseline.yaml --data "$DATA" \
    --profile colab_t4 --hub hf --resume auto --hf-repo "$HF_REPO" --epochs 15

In [ ]:
import json
r = json.load(open("results/phase1_baseline.json"))
print("test_acc   =", r["test_acc"])
print("macro_f1   =", r["macro_f1"])
print(r["confusion_matrix"])

## Results

- Final JSON: `/content/drive/MyDrive/cse475/results/phase1_baseline.json`
- Checkpoint already synced to HF every epoch.
- Next machine just runs with `--hub hf --resume auto` and picks up where Colab stopped.